# SCI1070 — Semaine 5 : le réseau Q profond, et ce que chaque réparation répare

| Volet | Ce que vous y faites | Statut |
| --- | --- | --- |
| A | un DQN complet sur `CartPole-v1`, des cinq pièces à la convergence | implémentation |
| B | **l'ablation : réseau cible et tampon, retirés un par un** | **évalué** |
| C | évaluer un agent de Pong à partir de poids fournis | récompense |

**Budget de calcul.** Un DQN complet sur CartPole tient en **moins de quinze
minutes** sur un T4. Un DQN sur Pong demande **une à trois heures** : il est hors de
portée d'une session, et le volet C **n'entraîne rien**.

> **Un conseil avant de lancer quoi que ce soit.** Activez l'accélérateur graphique
> dans Colab **avant** d'exécuter la première cellule. Chaque session où un étudiant
> se plaint d'une lenteur commence par la ligne ci-dessous, et elle affiche `False`
> neuf fois sur dix.

In [ ]:
import importlib.util
import subprocess
import sys

for paquet, module in [("gymnasium[classic-control]>=1.0", "gymnasium"), ("torch", "torch")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)

import collections

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

peripherique = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch.cuda.is_available() :", torch.cuda.is_available(), "| périphérique :", peripherique)

GAMMA = 0.99
BATCH_SIZE = 64
REPLAY_SIZE = 10_000
REPLAY_START_SIZE = 1_000
LEARNING_RATE = 1e-3
SYNC_TARGET_FRAMES = 500
EPSILON_DEBUT, EPSILON_FIN, EPSILON_TRAMES = 1.0, 0.02, 10_000
SEUIL_RESOLU = 475

Transition = collections.namedtuple(
    "Transition", ["obs", "action", "recompense", "termine", "obs_suivante"])

---
# Volet A : un DQN complet sur CartPole

CartPole n'a besoin d'**aucune** machinerie Atari : pas d'enveloppes d'image, pas de
convolutions, pas d'empilement de trames, puisque l'observation contient déjà la
vitesse et la vitesse angulaire.

In [ ]:
class TamponDeRejeu:
    def __init__(self, capacite, graine=0):
        self.memoire = ...        # À COMPLÉTER : une seule ligne, et elle contient maxlen
        self.rng = np.random.default_rng(graine)

    def __len__(self):
        return len(self.memoire)

    def ajouter(self, transition):
        self.memoire.append(transition)

    def _depaqueter(self, lot):
        return (np.asarray([t.obs for t in lot], dtype=np.float32),
                np.asarray([t.action for t in lot], dtype=np.int64),
                np.asarray([t.recompense for t in lot], dtype=np.float32),
                np.asarray([t.termine for t in lot], dtype=bool),
                np.asarray([t.obs_suivante for t in lot], dtype=np.float32))

    def echantillonner(self, taille_lot):
        ...        # À COMPLÉTER : tirage uniforme, puis cinq tableaux NumPy

    def dernieres(self, taille_lot):
        """AUCUN tirage : les transitions les plus récentes, dans l'ordre."""
        return self._depaqueter(list(self.memoire)[-taille_lot:])

In [ ]:
class ReseauQ(nn.Module):
    """Deux couches cachées de 128 unités. Une sortie par action, aucune
    non-linéarité après la dernière couche."""

    def __init__(self, n_obs, n_actions):
        super().__init__()
        ...        # À COMPLÉTER : le réseau de la section 5.5

    def forward(self, x):
        return ...        # À COMPLÉTER : un seul appel

In [ ]:
def calculer_perte(lot, reseau, reseau_cible, gamma, peripherique):
    obs, actions, recompenses, termines, obs_suivantes = lot

    obs_t = torch.as_tensor(obs, dtype=torch.float32).to(peripherique)
    obs_suiv_t = torch.as_tensor(obs_suivantes, dtype=torch.float32).to(peripherique)
    actions_t = torch.as_tensor(actions, dtype=torch.int64).to(peripherique)
    recompenses_t = torch.as_tensor(recompenses, dtype=torch.float32).to(peripherique)
    termines_t = torch.as_tensor(termines, dtype=torch.bool).to(peripherique)

    valeurs_action = ...        # À COMPLÉTER : gather sur la dimension des actions
    with torch.no_grad():
        valeurs_suivantes = ...        # À COMPLÉTER : le max, et sur QUEL réseau ?
        ...        # À COMPLÉTER : le masque terminal
    cibles = ...        # À COMPLÉTER : la cible complète
    return nn.MSELoss()(valeurs_action, cibles)

### Les quatre points de contrôle

Chacun se vérifie en une assertion, et ils attrapent quatre des cinq bogues
classiques de la semaine.

In [ ]:
# On fabrique un lot et deux réseaux pour pouvoir vérifier les quatre points.
reseau = ReseauQ(4, 2).to(peripherique)
reseau_cible = ReseauQ(4, 2).to(peripherique)

faux_lot = (np.zeros((BATCH_SIZE, 4), dtype=np.float32),
            np.zeros(BATCH_SIZE, dtype=np.int64),
            np.ones(BATCH_SIZE, dtype=np.float32),
            np.array([i % 2 == 0 for i in range(BATCH_SIZE)]),
            np.zeros((BATCH_SIZE, 4), dtype=np.float32))

obs_t = torch.as_tensor(faux_lot[0]).to(peripherique)
actions_t = torch.as_tensor(faux_lot[1]).to(peripherique)
termines_t = torch.as_tensor(faux_lot[3]).to(peripherique)
valeurs_action = reseau(obs_t).gather(1, actions_t.unsqueeze(-1)).squeeze(-1)
with torch.no_grad():
    valeurs_suivantes = reseau_cible(obs_t).max(1)[0]
    valeurs_suivantes[termines_t] = 0.0
cibles = torch.as_tensor(faux_lot[2]).to(peripherique) + GAMMA * valeurs_suivantes

# 1. gather prélève bien UNE valeur par ligne du lot.
assert valeurs_action.shape == (BATCH_SIZE,)

# 2. La cible ne porte AUCUN gradient. C'est le semi-gradient, vérifié.
assert not cibles.requires_grad

# 3. Les deux réseaux sont distincts en mémoire, et identiques après synchro.
assert reseau is not reseau_cible
reseau_cible.load_state_dict(reseau.state_dict())
assert torch.equal(next(reseau.parameters()), next(reseau_cible.parameters()))

# 4. Le masque terminal a bien mis des zéros, et pas des valeurs de table.
assert float(valeurs_suivantes[termines_t].abs().sum()) == 0.0

print("quatre points de contrôle passés")

### La boucle

Elle est paramétrée dès maintenant : le volet B va la rappeler **trois fois** en ne
changeant qu'une chose, et une boucle recopiée trois fois est la façon la plus sûre
de faire diverger ce qu'on croit comparer.

In [ ]:
def entrainer_dqn(sans_reseau_cible=False, sans_tampon=False, graine=0,
                  max_trames=60_000, env_id="CartPole-v1"):
    """Rend (trames, retours_par_episode, trames_cumulees)."""
    ...        # À COMPLÉTER : la boucle complète. `sans_reseau_cible` fait passer la cible par `reseau` ; `sans_tampon` prend les 64 DERNIÈRES transitions, dans l'ordre. Comptez les TRAMES.

In [ ]:
trames, retours, trames_cumulees = entrainer_dqn()
print(f"{trames} trames, {len(retours)} épisodes, "
      f"retour final sur 100 épisodes : {retours[-100:].mean():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(trames_cumulees, retours, linewidth=0.6)
axes[0].set_title("brut : un DQN est bruité")
lisse = np.convolve(retours, np.ones(10) / 10, mode="valid")
axes[1].plot(trames_cumulees[9:], lisse)
axes[1].set_title("lissé sur dix épisodes : il apprend quand même")
for ax in axes:
    ax.set_xlabel("trames")      # des TRAMES, jamais des épisodes
    ax.set_ylabel("retour de l'épisode")
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Attentes du volet A.**

* Résolu en **quelques dizaines de milliers de trames**, en moins de quinze minutes.
* Attendez-vous à voir le retour **redescendre** après un premier succès. Ce n'est
  pas un bogue, c'est le régime de la section 5.9.
* **Comptez les trames, pas les épisodes.** Si votre axe horizontal est en épisodes,
  votre courbe compare des épisodes de 20 pas avec des épisodes de 500 pas.

---
# Volet B, évalué : l'ablation

**Objectif. Mesurer, et non croire.** Trois configurations, **le même code**, **la
même graine**, et trois courbes sur un seul graphique.

| Configuration | La seule chose qui change | Ce que vous retirez |
| --- | --- | --- |
| **(1) complète** | rien | la référence du volet A |
| **(2) sans réseau cible** | `reseau_cible` remplacé par `reseau` | la cible cesse d'être figée |
| **(3) sans tampon** | les 64 **dernières** transitions, dans l'ordre | l'indépendance des données |

In [ ]:
configurations = {
    "(1) complète": dict(),
    "(2) sans réseau cible": dict(sans_reseau_cible=True),
    "(3) sans tampon": dict(sans_tampon=True),
}

resultats = {}
for nom, options in configurations.items():
    resultats[nom] = entrainer_dqn(graine=0, **options)    # LA MÊME GRAINE
    print(f"{nom:24s} {resultats[nom][1][-20:].mean():7.1f} sur les 20 derniers épisodes")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for nom, (trames, retours_c, cumul) in resultats.items():
    lisse = np.convolve(retours_c, np.ones(10) / 10, mode="valid")   # lissage IDENTIQUE
    ax.plot(cumul[9:], lisse, label=nom)
ax.set_xlabel("trames"); ax.set_ylabel("retour, lissé sur dix épisodes")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

# Le tableau à trois lignes : trames pour atteindre une moyenne de 200, ou « jamais ».
for nom, (trames, retours_c, cumul) in resultats.items():
    lisse = np.convolve(retours_c, np.ones(10) / 10, mode="valid")
    atteint = int(np.argmax(lisse >= 200)) if (lisse >= 200).any() else None
    quand = f"{cumul[9:][atteint]:,} trames" if atteint is not None else "jamais atteint"
    print(f"{nom:24s} {quand}")

**Attentes du volet B, et c'est ce que je corrigerai.**

* Les trois courbes sur un même graphique, axe horizontal en **trames**, **lissage
  identique pour les trois**. Sans cette dernière précaution, la comparaison ne veut
  rien dire.
* **Une demi-page écrite.** Pour chacune des deux ablations : quelle hypothèse de la
  section 5.2 ou 5.7 vous venez de violer, ce que vous observez, et pourquoi la
  seconde découle de la première. Les mots attendus : **« non stationnaire »**,
  **« corrélées »**, **« distribution »**.
  * Sans réseau cible : la cible bouge à chaque pas puisqu'elle sort du réseau qu'on
    entraîne. L'objectif devient **non stationnaire**, et la régression poursuit une
    cible qui fuit.
  * Sans tampon : les 64 dernières transitions viennent du même épisode et sont donc
    fortement **corrélées**. La **distribution** du lot n'est plus celle du
    comportement de l'agent, et chaque mise à jour tire le réseau vers le dernier
    bout de trajectoire.
* **Une mise en garde que vous écrivez vous-même** : une seule exécution par
  configuration ne suffit pas à conclure, parce que la dispersion entre graines est
  du même ordre que l'écart mesuré (section 5.9).

In [ ]:
...        # À COMPLÉTER : facultatif : la quatrième configuration, sans detach ni no_grad, et votre prédiction écrite

---
# Volet C : jouer à Pong, à partir de poids fournis

**Objectif.** Voir ce que la méthode produit, et écrire la boucle d'évaluation qui
le montre. **Rien n'est entraîné dans ce volet.**

### D'où viennent les poids, et pourquoi le dépôt du cours n'en livre aucun

Ce dépôt ne contient **aucun fichier de poids** : les points de contrôle publics qui
conviendraient sont publiés **sans licence déclarée**. Ce que le cours fournit, c'est
la **conversion**, et elle est testée. L'architecture de Stable-Baselines3 pour les
images **est** celle de la section 5.8, aux noms près.

**Trois routes :** entraîner vous-même et convertir ; télécharger un point de
contrôle public **après avoir vérifié sa licence**, puis convertir ; ou demander à
votre personne enseignante le fichier qu'elle met à disposition.

In [ ]:
for paquet, module in [("ale-py>=0.10", "ale_py"), ("opencv-python", "cv2")]:
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", paquet], check=True)

import ale_py
gym.register_envs(ale_py)

from sci1070.poids import convertir_sb3
from stable_baselines3 import DQN as DQN_SB3

modele_sb3 = DQN_SB3.load("dqn-PongNoFrameskip-v4.zip")
torch.save(convertir_sb3(modele_sb3.policy.state_dict()), "pong_poids.dat")
print("pong_poids.dat écrit")

In [ ]:
...        # À COMPLÉTER : la chaîne d'enveloppes de la section 5.8, dans le bon ordre. L'empilement de trames est le DERNIER maillon.

...        # À COMPLÉTER : le réseau DQN de la section 5.8 (convolutions + tete)

In [ ]:
reseau = DQN(env.observation_space.shape, env.action_space.n).to(peripherique)
reseau.load_state_dict(torch.load("pong_poids.dat", map_location=peripherique))
reseau.eval()                       # décision 1 : mode évaluation

N_EPISODES = 10
retours = []
for _ in range(N_EPISODES):
    obs, _ = env.reset()
    retour, termine, tronque = 0.0, False, False
    while not (termine or tronque):
        obs_t = torch.as_tensor(obs).unsqueeze(0).to(peripherique)
        with torch.no_grad():                        # décision 2 : aucun gradient
            action = int(reseau(obs_t).argmax(dim=1).item())   # PUREMENT glouton
        obs, recompense, termine, tronque, _ = env.step(action)
        retour += float(recompense)
    retours.append(retour)

print(f"retour moyen sur {N_EPISODES} épisodes : {np.mean(retours):.1f}")

**Attentes du volet C.**

* Un retour moyen **positif** sur une dizaine d'épisodes.
* **La question à traiter en trois lignes** : pourquoi l'évaluation se fait-elle avec
  \(\varepsilon = 0\), alors que l'entraînement gardait \(\varepsilon = 0{,}01\) ?
  Parce que la **politique apprise** est celle que l'on veut mesurer, et que la
  **politique jouée** pendant l'entraînement explore encore. C'est exactement la
  falaise de la semaine 4, sur un autre jeu.
* **La vérification qui attrape le bogue classique** : si votre agent obtient
  **−21 à tous les épisodes** avec des poids réputés bons, le problème est dans la
  **chaîne d'enveloppes**, pas dans les poids. Vérifiez dans l'ordre : la forme de
  l'observation, l'ordre des axes, l'appui sur FIRE, puis la position de l'enveloppe
  d'empilement, qui doit être la **dernière**.